# Data Exploration and Cleaning
### Goals:
1. Load and clean dataset and demographics
2. Confirm baseline EEG availability for labelled subjects
3. Characterise demographics and class balance for the resulting subset; verify the Responder outcome definition

Note:
1. Required inputs: `TDBRAIN_participants_V3.xlsx` and the `TDBRAIN_Dataset_V3_1` folder.
2. Baseline EEG session is assumed to be `sessID == 1`; not explicitly confirmed in TDBRAIN documentation.

In [1]:
# Import libraries 
import numpy as np
import pandas as pd 
from pathlib import Path
import glob        
import matplotlib.pyplot as plt
import seaborn as sns   
from scipy import stats                                                                                              

In [2]:
# load participants data from TDBRAIN dataset

#define function to find repo root
def find_repo_root(marker = ".git"): # .git marks the repo root reliably regardless of who runs this or from where
    current_path = Path.cwd()
    for parent in [current_path, *current_path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory of {current_path}")

project_root = find_repo_root()
data_dir = project_root / "data" 

TDBRAIN_participant_data = pd.read_excel(data_dir/"TDBRAIN_participants_V3.xlsx")

In [3]:
#Explore the data
print(TDBRAIN_participant_data.columns.tolist())
print(TDBRAIN_participant_data['indication'].value_counts())
print(TDBRAIN_participant_data['DISC/REP'].value_counts())
print(TDBRAIN_participant_data.shape)

['TDBRAIN_ID', 'DISC/REP', 'indication', 'formal_status', 'Dataset', 'Consent', 'sessSeason', 'sessTime', 'Responder', 'Remitter', 'age', 'gender', 'sessID', 'nrSessions', 'neoFFI_q1', 'neoFFI_q2', 'neoFFI_q3', 'neoFFI_q4', 'neoFFI_q5', 'neoFFI_q6', 'neoFFI_q7', 'neoFFI_q8', 'neoFFI_q9', 'neoFFI_q10', 'neoFFI_q11', 'neoFFI_q12', 'neoFFI_q13', 'neoFFI_q14', 'neoFFI_q15', 'neoFFI_q16', 'neoFFI_q17', 'neoFFI_q18', 'neoFFI_q19', 'neoFFI_q20', 'neoFFI_q21', 'neoFFI_q22', 'neoFFI_q23', 'neoFFI_q24', 'neoFFI_q25', 'neoFFI_q26', 'neoFFI_q27', 'neoFFI_q28', 'neoFFI_q29', 'neoFFI_q30', 'neoFFI_q31', 'neoFFI_q32', 'neoFFI_q33', 'neoFFI_q34', 'neoFFI_q35', 'neoFFI_q36', 'neoFFI_q37', 'neoFFI_q38', 'neoFFI_q39', 'neoFFI_q40', 'neoFFI_q41', 'neoFFI_q42', 'neoFFI_q43', 'neoFFI_q44', 'neoFFI_q45', 'neoFFI_q46', 'neoFFI_q47', 'neoFFI_q48', 'neoFFI_q49', 'neoFFI_q50', 'neoFFI_q51', 'neoFFI_q52', 'neoFFI_q53', 'neoFFI_q54', 'neoFFI_q55', 'neoFFI_q56', 'neoFFI_q57', 'neoFFI_q58', 'neoFFI_q59', 'neoFFI_q60

In [4]:
#Filter data set to include only MDD patients with response to rTMS treatment data
mdd_labelled = TDBRAIN_participant_data[
    (TDBRAIN_participant_data['indication'] == 'MDD') &
    (TDBRAIN_participant_data['Responder'].notna())
].copy()

# Keep the earliest session per subject as the assumed baseline
# (TDBRAIN documentation does not explicitly confirm sessID==1 = pre-treatment,
# but session numbering is presumed chronological - see README)
mdd_labelled = mdd_labelled.sort_values('sessID').drop_duplicates(subset='TDBRAIN_ID', keep='first')
mdd_labelled.shape

(163, 106)

In [5]:
# Check responders vs non-responders distribution
print(mdd_labelled['Responder'].value_counts())

Responder
1.0    94
0.0    69
Name: count, dtype: int64


In [6]:
#Check which participants have resting state EEG data (eyes closed and eyes open)

discovery_dir = data_dir / "TDBRAIN_Dataset_V3_1"                                                                                                                        

has_ec = []
has_eo = []   

for sid in mdd_labelled['TDBRAIN_ID']:
  ec_files = glob.glob(f'{discovery_dir}/{sid}/ses-*/eeg/*_task-restEC_eeg.bdf')
  eo_files = glob.glob(f'{discovery_dir}/{sid}/ses-*/eeg/*_task-restEO_eeg.bdf')
  has_ec.append(len(ec_files) > 0)
  has_eo.append(len(eo_files) > 0)

mdd_labelled['has_restEC'] = has_ec
mdd_labelled['has_restEO'] = has_eo

print(mdd_labelled[['TDBRAIN_ID', 'has_restEC', 'has_restEO']].to_string())
print()
print('restEC:', sum(has_ec), '/', len(has_ec))
print('restEO:', sum(has_eo), '/', len(has_eo))

        TDBRAIN_ID  has_restEC  has_restEO
320   sub-87999321        True        True
1008  sub-88049537        True        True
1015  sub-88049857        True        True
1016  sub-88049905        True        True
1027  sub-88050713        True        True
1034  sub-88052013        True        True
1036  sub-88052061        True        True
1041  sub-88052329        True        True
1043  sub-88052465        True        True
1049  sub-88052869        True        True
1055  sub-88053317        True        True
1064  sub-88053997        True        True
1070  sub-88054313        True        True
1071  sub-88054357        True        True
1075  sub-88054577        True        True
1082  sub-88055749        True        True
1085  sub-88056017        True        True
1100  sub-88057913        True        True
1006  sub-88049405        True        True
1102  sub-88058229        True        True
999   sub-88048817        True        True
972   sub-88047245        True        True
883   sub-8

In [7]:
#multi session check - how many participants have 2 sessions?
print(mdd_labelled['nrSessions'].value_counts())
multi_session = mdd_labelled[mdd_labelled['nrSessions'] == 2]
print(multi_session[['TDBRAIN_ID', 'sessID', 'nrSessions', 'sessSeason', 'sessTime']].to_string())
multi_sess_unfiltered = TDBRAIN_participant_data[(TDBRAIN_participant_data['indication'] == 'MDD') & (TDBRAIN_participant_data['nrSessions'] == 2)]
print(multi_sess_unfiltered[['TDBRAIN_ID', 'sessID', 'nrSessions', 'sessSeason', 'Responder']].to_string())

nrSessions
1    155
2      8
Name: count, dtype: int64
       TDBRAIN_ID  sessID  nrSessions sessSeason sessTime
942  sub-88045713       1           2       fall      NaN
442  sub-88010981       1           2     spring      NaN
472  sub-88012461       1           2     summer      NaN
568  sub-88017409       1           2       fall      NaN
344  sub-88006161       1           2     winter      NaN
420  sub-88009901       1           2     spring      NaN
651  sub-88022089       1           2     spring      NaN
665  sub-88023125       1           2     spring      NaN
       TDBRAIN_ID  sessID  nrSessions sessSeason  Responder
344  sub-88006161       1           2     winter        1.0
345  sub-88006161       2           2     summer        1.0
420  sub-88009901       1           2     spring        1.0
421  sub-88009901       2           2     winter        1.0
442  sub-88010981       1           2     spring        1.0
443  sub-88010981       2           2       fall        1.0
472

163/163 subjects have baseline restEC/restEO; 8 subjects have multiple sessions; sessID==1 taken as baseline

## Demographics characterisation
1. Compare Responder vs Non-responder on age and gender. 
2. Check whether Responders and Non-responders started at a similar baseline BDI severity. 
3. Verify the Responder definition itself

In [8]:
##1
# Get summary stats for age and gender grouped by responder status
age_stats = mdd_labelled.groupby('Responder')['age'].agg(['mean', 'std'])
print(age_stats)

gender_props = mdd_labelled.groupby('Responder')['gender'].value_counts(normalize=True)
print(gender_props)

                mean        std
Responder                      
0.0        48.678116  13.754458
1.0        42.849681  13.619650
Responder  gender
0.0        1         0.565217
           0         0.434783
1.0        0         0.542553
           1         0.457447
Name: proportion, dtype: float64


In [9]:
# run Welch's t-test between the two groups' ages to check whether the difference in mean age is statistically significant
t_stat, p_value = stats.ttest_ind(
    mdd_labelled[mdd_labelled['Responder'] == 1]['age'].dropna(),
    mdd_labelled[mdd_labelled['Responder'] == 0]['age'].dropna(),
    equal_var=False
)
print('Difference between mean ages of responders and non-responders:')
print(f'T-statistic: {t_stat:.2f}')
print(f'P-value: {p_value:.2f}')

# run chi-squared test to check whether the difference in gender distribution between responders and non-responders is statistically significant
contingency_table = pd.crosstab(mdd_labelled['Responder'], mdd_labelled['gender'])
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print('Difference between gender distribution of responders and non-responders:')
print(f'Chi-squared statistic: {chi2_stat:.2f}')
print(f'P-value: {p_value:.2f}')

Difference between mean ages of responders and non-responders:
T-statistic: -2.68
P-value: 0.01
Difference between gender distribution of responders and non-responders:
Chi-squared statistic: 1.44
P-value: 0.23


In [10]:
##2
bdi_pre_stats = mdd_labelled.groupby('Responder')['BDI_pre'].agg(['mean', 'std'])
print(bdi_pre_stats)
t_stat, p_value = stats.ttest_ind(
    mdd_labelled[mdd_labelled['Responder'] == 1]['BDI_pre'].dropna(),
    mdd_labelled[mdd_labelled['Responder'] == 0]['BDI_pre'].dropna(),
    equal_var=False
)
print(f'T-statistic: {t_stat:.2f}, P-value: {p_value:.2f}')

                mean        std
Responder                      
0.0        33.362319  10.509059
1.0        31.000000   9.940684
T-statistic: -1.45, P-value: 0.15


Summary of satistical tests:
1. Age differs significantly (confound to flag)
2. Gender is balanced
3. Baseline BDI severity balanced.

In [11]:
##3
# compute percent change from BDI_pre to BDI_post for each subject
#Filter to subjects with non-null BDI_pre and BDI_post
mdd_BDI_filtered = mdd_labelled[mdd_labelled['BDI_pre'].notna() & mdd_labelled['BDI_post'].notna()].copy()
print(f'Number of subjects with non-null BDI_pre and BDI_post: {len(mdd_BDI_filtered)}')
#Compute percent improvement using: -(BDI_post - BDI_pre) / BDI_pre * 100
def compute_percent_improvement(row):
    if row['BDI_pre'] == 0:
        return np.nan  # Avoid division by zero
    return -(row['BDI_post'] - row['BDI_pre']) / row['BDI_pre'] * 100

mdd_BDI_filtered['percent_improvement'] = mdd_BDI_filtered.apply(compute_percent_improvement, axis=1)
print(mdd_BDI_filtered[['TDBRAIN_ID', 'BDI_pre', 'BDI_post', 'percent_improvement']].to_string())
print(mdd_BDI_filtered['percent_improvement'].describe())
# see whether a 50% threshold actually lines up with the existing Responder label. 
derived_responder = (mdd_BDI_filtered['percent_improvement'] >= 50).astype(int)
#Compare derived_responder against the existing Responder column
comparison = pd.DataFrame({
    'TDBRAIN_ID': mdd_BDI_filtered['TDBRAIN_ID'],
    'existing_responder': mdd_BDI_filtered['Responder'],
    'derived_responder': derived_responder
})
print(comparison.to_string())
print('Number of subjects where existing and derived responder labels match:', (comparison['existing_responder'] == comparison['derived_responder']).sum())
print('Number of subjects where existing and derived responder labels do not match:', (comparison['existing_responder'] != comparison['derived_responder']).sum())


Number of subjects with non-null BDI_pre and BDI_post: 163
        TDBRAIN_ID  BDI_pre  BDI_post  percent_improvement
320   sub-87999321     20.0       8.0            60.000000
1008  sub-88049537     23.0       0.0           100.000000
1015  sub-88049857     52.0      25.0            51.923077
1016  sub-88049905     36.0       3.0            91.666667
1027  sub-88050713     41.0      13.0            68.292683
1034  sub-88052013     30.0      12.0            60.000000
1036  sub-88052061     42.0      14.0            66.666667
1041  sub-88052329     50.0      44.0            12.000000
1043  sub-88052465     37.0      26.0            29.729730
1049  sub-88052869     26.0      24.0             7.692308
1055  sub-88053317     23.0      16.0            30.434783
1064  sub-88053997     31.0       9.0            70.967742
1070  sub-88054313     19.0      27.0           -42.105263
1071  sub-88054357     32.0      11.0            65.625000
1075  sub-88054577     51.0      30.0            41.1764

# Notebook '1_cohort_exploration' summary
This notebook started by loading the demographics data and characterising missingness and classes balance to spot potential confounds. 
The final sample size that remains of MDD patients with rTMS response documentation is 163. 
Note: 
1. Responder status as >50% BDI improvement verified (163/163 match)
2. Similar MDD baseline severity between responders and non-responders verified (no significant difference)
3. One potential confound identified: age (significant, p=.01) — should be controlled for in modelling
4. sessID==1 taken as baseline where 2 sessions exist (documented assumption, not confirmed in TDBRAIN documentation)
5. No missingness in the fields used in this project (age, gender, BDI_pre, BDI_post, Responder) - confirmed via demographics and BDI verification steps above; no additional covariates (e.g. education, NEO-FFI) are in scope for this project